# Step 2: Manual Cell Selection and Label Correction with Napari

**Purpose:** After Cellpose segmentation, use this notebook to specify which cells to include in the analysis and to fix any masks that are missing or defective in individual frames.  Run it for every subject after `01_automatic_segmentation.ipynb`.

**Run order within `01-preprocess/`:**
```
01_automatic_segmentation.ipynb   ← always run first
02_manual_labelling.ipynb         ← always run after (this notebook)
cyh_03_image_registration.py      ← always run last
```

**Prerequisites:**
- Cellpose masks must already exist for the subject (`sub-XXX_data-nucMask.tif`, `sub-XXX_data-memMask.tif`).
- The `utility` package must be importable (add its parent directory to `PYTHONPATH` if needed).
- Napari must be installed in the active environment.

**Outputs written to `D:/Kou - Trunk Neural Crest 2D/project-01/sub-XXX/`:**
- `sub-XXX_data-nucManualLabel.tif` — manually specified/corrected nucleus labels (uint16)
- `sub-XXX_data-memManualLabel.tif` — manually specified/corrected membrane labels (uint16)

## 1 · Imports and helper functions

In [ ]:
import os
import sys

import napari
import numpy as np
from matplotlib import cm
from tifffile import imread, imwrite

sys.path.append("..")          # make `utility` importable from the repo root
from utility import config

In [ ]:
# ---------------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------------

def _blue_colormap(label_array: np.ndarray) -> dict:
    """Return a Napari color dict mapping existing label IDs to shades of blue.

    Label 0 and None are set to fully transparent so the background is invisible.
    The gradient runs from light blue (label 1) to dark blue (highest label).
    """
    n = int(label_array.max()) + 1
    blues = cm.Blues(np.linspace(0.4, 1.0, n))   # lighter → darker
    color_dict = {i: tuple(blues[i][:3]) for i in range(n)}
    color_dict[0]    = (0, 0, 0, 0)   # background: transparent
    color_dict[None] = (0, 0, 0, 0)   # napari sentinel: transparent
    return color_dict


def add_label_layer(viewer: napari.Viewer, label_data: np.ndarray, layer_name: str):
    """Add a label layer to *viewer* and configure it for manual annotation.

    Existing Cellpose labels are shown in shades of blue.
    New labels drawn by the user start at ``max_existing_label + 1`` and cycle
    through a set of high-contrast colours so hand-drawn regions are easy to
    distinguish from the automatic segmentation.

    Parameters
    ----------
    viewer     : active Napari viewer
    label_data : uint16 label array (frames × height × width)
    layer_name : string displayed in the Napari layer list

    Returns
    -------
    label_layer : napari Labels layer (holds a reference for later saving)
    max_label   : highest label ID present before any manual edits
    """
    colors = _blue_colormap(label_data)
    max_label = int(np.max(label_data))

    # Reserve slots above max_label for user-drawn annotations.
    # The selected_label is set to max_label+1 so the first brush stroke
    # immediately uses the first annotation colour.
    annotation_colors = {
        max_label +  1: "red",     max_label +  2: "orange",
        max_label +  3: "yellow",  max_label +  4: "green",
        max_label +  5: "pink",    max_label +  6: "blue",
        max_label +  7: "purple",  max_label +  8: "brown",
        max_label +  9: "cyan",    max_label + 10: "magenta",
        max_label + 11: "lime",    max_label + 12: "olive",
        max_label + 13: "navy",    max_label + 14: "teal",
        max_label + 15: "maroon",  max_label + 16: "silver",
        max_label + 17: "gold",    max_label + 18: "salmon",
        max_label + 19: "plum",
    }
    colors.update(annotation_colors)

    label_layer = viewer.add_labels(
        label_data,
        name=layer_name,
        opacity=0.2,
        colormap=colors,
    )
    label_layer.selected_label = max_label + 1   # start brush on first annotation slot
    return label_layer, max_label


def save_manual_labels(
    label_layer,
    max_auto_label: int,
    sub_id: int,
    project_dir: str,
    label_type: str = "nuc",
) -> None:
    """Strip the original Cellpose labels and save only the manually drawn regions.

    Any pixel whose label ID is <= *max_auto_label* was part of the automatic
    segmentation and is zeroed out.  The remaining (user-drawn) labels are
    re-indexed to consecutive integers starting at 1 before saving.

    Parameters
    ----------
    label_layer    : Napari Labels layer returned by ``add_label_layer``
    max_auto_label : highest label ID that existed before manual editing
    sub_id         : integer subject ID (used to construct the output filename)
    project_dir    : root data directory (e.g. config.data_dir)
    label_type     : ``'nuc'`` or ``'mem'`` — selects the output filename suffix
    """
    label_data = label_layer.data.copy()

    # Zero out every pixel that belongs to an auto-segmentation label
    label_data[label_data <= max_auto_label] = 0

    # Re-index surviving labels to 1, 2, 3, …
    unique_labels = np.unique(label_data[label_data > 0])   # skip background (0)
    for new_id, old_id in enumerate(unique_labels, start=1):
        label_data[label_data == old_id] = new_id

    out_path = os.path.join(
        project_dir,
        f"sub-{sub_id:03d}",
        f"sub-{sub_id:03d}_data-{label_type}ManualLabel.tif",
    )
    imwrite(out_path, label_data.astype(np.uint16))
    print(f"Saved → {out_path}  ({len(unique_labels)} manual label(s))")

## 2 · Parameters

Set `sub_id` to the subject you want to correct.  Everything else is derived from `config.data_dir`.

In [ ]:
# ── USER INPUT ──────────────────────────────────────────────────────────────
sub_id = 17          # zero-padded to 3 digits automatically (e.g. 17 → 'sub-017')
# ────────────────────────────────────────────────────────────────────────────

project_dir = config.data_dir
sub_dir     = os.path.join(project_dir, f"sub-{sub_id:03d}")

paths = {
    "xyProjection": os.path.join(sub_dir, f"sub-{sub_id:03d}_data-xyProjection.tif"),
    "nucMask":      os.path.join(sub_dir, f"sub-{sub_id:03d}_data-nucMask.tif"),
    "memMask":      os.path.join(sub_dir, f"sub-{sub_id:03d}_data-memMask.tif"),
}

# Verify all required files exist before proceeding
missing = [k for k, p in paths.items() if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(
        f"Missing files for sub-{sub_id:03d}: {missing}\n"
        "Run 01_automatic_segmentation.ipynb first."
    )
print(f"All input files found for sub-{sub_id:03d}.")

## 3 · Load data

In [ ]:
print("Loading masks and projection…")
nuc_label    = imread(paths["nucMask"])       # shape: (frames, H, W)
mem_label    = imread(paths["memMask"])
xy_proj      = imread(paths["xyProjection"])  # shape: (frames, 2, H, W)

img_mem = xy_proj[:, 0, :, :]   # channel 0 = membrane
img_nuc = xy_proj[:, 1, :, :]   # channel 1 = nucleus

print(
    f"  Projection shape : {xy_proj.shape}  (frames × channels × H × W)\n"
    f"  Nucleus mask     : {nuc_label.shape}, "
    f"{int(nuc_label.max())} labels, dtype={nuc_label.dtype}\n"
    f"  Membrane mask    : {mem_label.shape}, "
    f"{int(mem_label.max())} labels, dtype={mem_label.dtype}"
)

## 4 · Open Napari and draw corrections

Running the cell below launches the Napari viewer with four layers:

| Layer | Description |
|---|---|
| `nucProjection` | Raw nucleus channel (gray) |
| `memProjection` | Raw membrane channel (gray) |
| `nucLabel` | Nucleus Cellpose masks (blue gradient) |
| `memLabel` | Membrane Cellpose masks (blue gradient) |

**How to add a manual correction:**
1. Select the `nucLabel` or `memLabel` layer in the layer list.
2. Choose the **Paint brush** tool (keyboard shortcut `2`).
3. The active label is pre-set to `max_existing_label + 1` (shown in a distinct colour).
4. Paint over the region that needs to be added.  Each new label ID gets its own colour.
5. To label a new, separate region, increment the label ID in the label spinbox.
6. When finished, run the **Save** cell (section 5) below (you do *not* need to close Napari first).

In [ ]:
print("Opening Napari viewer…")
viewer = napari.Viewer()

# Add raw images for reference (additive blending lets both channels be visible)
viewer.add_image(img_nuc, name="nucProjection", colormap="gray", blending="additive")
viewer.add_image(img_mem, name="memProjection", colormap="gray", blending="additive")

# Add segmentation masks; store layer references and pre-edit max IDs for saving
nuc_label_layer, nuc_max_label = add_label_layer(viewer, nuc_label, "nucLabel")
mem_label_layer, mem_max_label = add_label_layer(viewer, mem_label, "memLabel")

print(
    f"Viewer ready.  Draw corrections, then run the Save cell below.\n"
    f"  Nucleus  — {nuc_max_label} existing labels; new labels start at {nuc_max_label + 1}\n"
    f"  Membrane — {mem_max_label} existing labels; new labels start at {mem_max_label + 1}"
)

## 5 · Save manual labels

Run this cell after you have finished drawing in Napari.  It will:
1. Extract only the pixels you painted (stripping the original Cellpose labels).
2. Re-index the remaining labels to consecutive integers starting at 1.
3. Write `sub-XXX_data-nucManualLabel.tif` and `sub-XXX_data-memManualLabel.tif`.

> **Note:** If you have not drawn anything in a given channel, the output file for that channel will contain only zeros — that is fine and expected.

In [ ]:
save_manual_labels(nuc_label_layer, nuc_max_label, sub_id, project_dir, label_type="nuc")
save_manual_labels(mem_label_layer, mem_max_label, sub_id, project_dir, label_type="mem")

print("\nDone.  You can now close the Napari viewer and proceed to:")
print("  cyh_03_image_registration.py")